<a href="https://colab.research.google.com/github/khoirulanamid/Upscale_Real_ESRGAN/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4k Video Upscaler Colab (Real-ESRGAN)

Adapted from: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN)

Made with ❤️ by: [yuvraj108c](https://github.com/yuvraj108c)

Github repository: https://github.com/yuvraj108c/4k-video-upscaler-colab

# 1. Setup (~1 minute)

In [1]:
import torch
assert torch.cuda.is_available(), "GPU not detected.. Please change runtime to GPU"

from PIL import Image
import cv2, os, subprocess
from tqdm import tqdm

# Clone repo if not exists
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

# Install dependencies with compatible versions
!pip install -q basicsr facexlib gfpgan ffmpeg-python
!pip install -q -r requirements.txt
!python setup.py develop

# Real-ESRGAN/BasicSR is currently incompatible with Numpy 2.0+
!pip install -q "numpy<2"

# --- PATCH FOR BASICSR COMPATIBILITY (Fixed Method) ---
import site
packages_path = site.getsitepackages()[0]
file_to_patch = os.path.join(packages_path, 'basicsr/data/degradations.py')

if os.path.exists(file_to_patch):
    with open(file_to_patch, 'r') as f:
        content = f.read()
    # Replace the deprecated import
    if 'torchvision.transforms.functional_tensor' in content:
        content = content.replace('from torchvision.transforms.functional_tensor import rgb_to_grayscale', 'from torchvision.transforms.functional import rgb_to_grayscale')
        with open(file_to_patch, 'w') as f:
            f.write(content)
        print("✅ Applied torchvision patch to basicsr.")
else:
    print("⚠️ Could not find basicsr degradations.py to patch. If errors persist, check installation.")

print("✅ Setup finished. Please restart the runtime if prompted!")

/content/Real-ESRGAN
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for d


# 2. Mount drive (optional)

In [2]:
from google.colab import drive
mount_drive=True #@param{type:"boolean"}

if mount_drive:
  drive.mount('/content/gdrive/')

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


# 3. Upscale video

- The upscaled video will be saved to `output_dir`
- If google drive is mounted, it will be also saved at `MyDrive/Upscaled Videos (REAL-ESRGAN)`


In [3]:
# @title 🚀 Upscale Video (Double-click to show code)
import os, subprocess, cv2
from google.colab import drive

# @markdown ### 📂 Batch Processing Folders
# @markdown Paste the path to your input folder from the left sidebar (Right-click folder -> Copy path)
input_folder = "/content/gdrive/MyDrive/Upscale_Input" # @param {type:"string"}
# @markdown Paste the path where you want to save the upscaled videos
output_dir = "/content/gdrive/MyDrive/Upscale_Output" # @param {type:"string"}

# @markdown ### ⚙️ Upscale Settings
resolution = "4k (3840 x 2160)" # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"] {type:"string"}
model = "RealESRGAN_x4plus" # @param ["RealESRGAN_x4plus" , "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
# @markdown Select output frame rate
fps_option = "60" # @param ["30", "60"] {type:"string"}
# @markdown Check this to remove audio from the final video
mute_video = True # @param {type:"boolean"}
# @markdown Skip videos that already exist in the output folder
auto_resume = True # @param {type:"boolean"}

# Ensure Drive is mounted if using gdrive paths
if "/content/gdrive" in input_folder or "/content/gdrive" in output_dir:
    if not os.path.exists("/content/gdrive"):
        drive.mount("/content/gdrive/")

os.makedirs(output_dir, exist_ok=True)

# Get list of video files
video_extensions = (".mp4", ".mkv", ".mov", ".avi")
if not os.path.exists(input_folder):
    print(f"❌ Error: Input folder '{input_folder}' not found!")
else:
    videos = [f for f in os.listdir(input_folder) if f.lower().endswith(video_extensions)]
    print(f"✅ Found {len(videos)} videos in {input_folder}")

    for video_file in videos:
        video_path = os.path.join(input_folder, video_file)
        video_name = os.path.splitext(video_file)[0]

        # Pre-calculate dimensions for filename check
        video_capture = cv2.VideoCapture(video_path)
        video_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
        video_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
        video_capture.release()

        final_width, final_height = 0, 0
        match resolution:
            case "FHD (1920 x 1080)": final_width, final_height = 1920, 1080
            case "2k (2560 x 1440)": final_width, final_height = 2560, 1440
            case "4k (3840 x 2160)": final_width, final_height = 3840, 2160
            case "2 x original": final_width, final_height = 2*video_width, 2*video_height
            case "3 x original": final_width, final_height = 3*video_width, 3*video_height
            case "4 x original": final_width, final_height = 4*video_width, 4*video_height

        final_video_name = f"{video_name}_upscaled_{final_width}_{final_height}_{fps_option}fps.mp4"
        final_video_path = os.path.join(output_dir, final_video_name)

        # --- AUTO-RESUME CHECK ---
        if auto_resume and os.path.exists(final_video_path):
            print(f"⏩ Skipping: {final_video_name} (Already exists)")
            continue

        print(f"\n--- 🚀 Processing: {video_file} ---")

        aspect_ratio = float(video_width/video_height)
        if aspect_ratio == 1.0 and "original" not in resolution: final_height = final_width
        if aspect_ratio < 1.0 and "original" not in resolution: final_width, final_height = final_height, final_width

        scale_factor = max(final_width/video_width, final_height/video_height)

        # Temp local output for processing speed
        temp_output = "/content/temp_upscaled/"
        os.makedirs(temp_output, exist_ok=True)

        # Run Inference
        !python inference_realesrgan_video.py -n {model} -i '{video_path}' -o '{temp_output}' --outscale {scale_factor}

        upscaled_video_path = os.path.join(temp_output, f"{video_name}_out.mp4")

        # Audio and FPS flags
        audio_flag = "-an" if mute_video else "-c:a copy"
        fps_flag = f"-r {fps_option}"

        # Finalize and move to destination
        if "original" not in resolution:
            print(f"✂️ Cropping and saving {final_video_name}...")
            command = f"ffmpeg -loglevel error -hwaccel cuda -y -i '{upscaled_video_path}' {fps_flag} {audio_flag} -c:v h264_nvenc -filter:v 'crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2' -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
            subprocess.run(command, shell=True)
        else:
            print(f"📦 Saving {final_video_name}...")
            command = f"ffmpeg -loglevel error -y -i '{upscaled_video_path}' {fps_flag} {audio_flag} -c:v libx264 -pix_fmt yuv420p '{final_video_path}'"
            subprocess.run(command, shell=True)

        # Cleanup
        if os.path.exists(upscaled_video_path): os.remove(upscaled_video_path)
        print(f"✨ Finished: {final_video_name}")

    print("\n🏁 Batch processing complete!")

✅ Found 2 videos in /content/gdrive/MyDrive/Upscale_Input

--- 🚀 Processing: Veo3-video-1776420836241-ry1onw26m.mp4 ---
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth

100% 63.9M/63.9M [00:00<00:00, 349MB/s]
inference: 100% 192/192 [19:28<00:00,  6.09s/frame]
✂️ Cropping and saving Veo3-video-1776420836241-ry1onw26m_upscaled_3840_2160.mp4...
✨ Finished: Veo3-video-1776420836241-ry1onw26m_upscaled_3840_2160.mp4

--- 🚀 Processing: Veo3-video-1776421108620-rr910nwfv.mp4 ---
inference: 100% 192/192 [19:52<00:00,  6.21s/frame]
✂️ Cropping and saving Veo3-video-1776421108620-rr910nwfv_upscaled_3840_2160.mp4...
✨ Finished: Veo3-video-1776421108620-rr910nwfv_upscaled_3840_2160.mp4

🏁 Batch processing complete!


# 4. Disconnect runtime

In [4]:
from google.colab import runtime

disconnect_when_finish = False  #@param{type:"boolean"}

if disconnect_when_finish:
  runtime.unassign()